# Setup

In [1]:
%pip install numpy pandas kagglehub seqeval datasets transformers torch python-dotenv tqdm matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import kagglehub
import os
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
import torch
from llm_api import OpenRouterApi
from dotenv import load_dotenv, dotenv_values 
import os
import json
from tqdm import tqdm
from typing import Tuple
import random

load_dotenv()

/home/joachim/Desktop/University-NER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Params

In [ ]:
DATASET_DIR = './data'

# OUTPUT_FILE = "hy3_results.jsonl"
# LLM_API = OpenRouterApi(key = os.getenv("OPEN_ROUTER_KEY"), model="tencent/hy3-preview:free")

OUTPUT_FILE = "deepseek_flash_results.jsonl"
LLM_API = OpenRouterApi(key = os.getenv("OPEN_ROUTER_KEY"), model="deepseek/deepseek-v4-flash")

TEMPERATURE: float = 0.2
MAX_TOKENS: int|None = 1024

EXAMPLES_AMT = 2
LLM_BASELINE_SUBSET = 500
TEST_SUBSET = 200

PROMPT = """
   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.
    
    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Example:
    [EXAMPLE]

    Now process the following input.

    Input:
    [INPUT]
"""

MAX_FAILS_AMT = 5

# Test API

In [4]:
LLM_API.send_message("Do you work correctly? Talk in english")

"Yes, I work correctly! I'm designed to understand and generate English text accurately, answer questions, assist with tasks, and hold conversations. If you ever encounter an issue or have a specific test in mind, feel free to ask—I’m here to help. How can I assist you today?"

# Dataset handling

In [5]:
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion", output_dir=DATASET_DIR)
print(path)
print(os.listdir(path))

./data
['valid.txt', 'test.txt', 'metadata', 'train.txt', '.complete']


In [6]:
def load_conll_ner(file_path):
    sentences = []
    tokens, labels = [], []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": labels})
                    tokens, labels = [], []
                continue
            
            if line.startswith("-DOCSTART-"):
                continue
            
            word, pos, chunk, ner = line.split()
            tokens.append(word)
            labels.append(ner)
    if tokens:
        sentences.append({"tokens": tokens, "ner_tags": labels})

    return sentences


train_data = load_conll_ner(f"{path}/train.txt")
val_data = load_conll_ner(f"{path}/valid.txt")
test_data  = load_conll_ner(f"{path}/test.txt")

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Number of training sentences: {len(train_dataset)}")
print(f"Number of validation sentences: {len(val_dataset)}")
print(f"Number of test sentences: {len(test_dataset)}")

print("Example sentence:")
print(train_dataset[0])

Number of training sentences: 14041
Number of validation sentences: 3250
Number of test sentences: 3453
Example sentence:
{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


# Training/testing loop

In [7]:
def create_example_json(min_length_of_example:int = 3) -> str:
    filtered = [ex for ex in train_dataset if len(ex["tokens"]) >= min_length_of_example]
    sorted_dataset = sorted(filtered, key=lambda ex: len(ex["tokens"]))
    shortest_pool = sorted_dataset[:EXAMPLES_AMT * 3]
    examples = random.sample(shortest_pool, EXAMPLES_AMT)

    example_str = ''
    for idx, ex in enumerate(examples):
        input_ = ex["tokens"]
        output_ = ex["ner_tags"]
        example_str += f'''
        {idx+1})
        Input:
        {json.dumps(input_, indent=None)}
        Output:
        {json.dumps(output_, indent=None)}
        '''

    return example_str

example_json = create_example_json()
print("Example JSON:")
print(example_json)

Example JSON:

        1)
        Input:
        ["68", "Steve", "Stricker"]
        Output:
        ["O", "B-PER", "I-PER"]
        
        2)
        Input:
        ["The", "media", "..."]
        Output:
        ["O", "O", "O"]
        


In [8]:
def prepare_prompt_and_sample(sample_idx, is_print=False) -> str:
    sample = test_dataset[sample_idx]
    prompt = PROMPT.replace("[EXAMPLE]", example_json).replace("[INPUT]", json.dumps(sample["tokens"], indent=None))
    
    if sample_idx == 0 and is_print:
        print("=======================================================")
        print(prompt)
        print("=======================================================\n")
    
    return prompt, sample


def parse_response(sample_input, chat_response) -> Tuple[list, str]:
    try:
        pred = json.loads(chat_response)
    except json.JSONDecodeError:
        return None, "Can not parse response to json"
    
    if not isinstance(pred, list):
        return None, "Response is not a JSON array"
    
    if not all(isinstance(t, str) for t in pred):
        return None, "Response contains non-string elements"
    
    if len(pred) != len(sample_input["tokens"]):
        return None, f"Predicted tags for sentence do not match tokens length"
        
    return pred, ""


print("Test prompt preparation:")
prompt, sample_input = prepare_prompt_and_sample(0, is_print=True)

Test prompt preparation:

   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.

    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Example:
    
        1)
        Input:
        ["68", "Steve", "Stricker"]
        Output:
        ["O", "B-PER", "I-PER"]
        
        2)
        Input:
        ["The", "media", "..."]
        Output:
        ["O", "O", "O"]
        

    Now process the following input.

    Input:
    ["SOCCER", "-", "JAPAN", "GET", "LUCKY", "WIN", ",", "CHINA", "IN", "SURPRISE", "DEFEAT", "."]




In [9]:
def try_load_results_from_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f]
    return []

In [10]:
results = try_load_results_from_file(OUTPUT_FILE)
pbar = tqdm(range(len(results), LLM_BASELINE_SUBSET))

for beg_idx in pbar:
    pbar.set_description(f"Processing sentence {beg_idx}")
    
    prompt, sample_input = prepare_prompt_and_sample(beg_idx)

    fails_amt = 0
    while True:
        response = LLM_API.send_message(prompt, 
                                        is_reasoning=fails_amt >= 2, 
                                        temperature=TEMPERATURE, 
                                        max_tokens=MAX_TOKENS * (fails_amt+1) if fails_amt < 2 else None,
                                        attempts_amt = 15)
        response_parsed, message = parse_response(sample_input, response)
        
        if response_parsed is None:
            fails_amt += 1
            
            if fails_amt >= MAX_FAILS_AMT:
                item = {"tokens": ["N/A"], 
                        "true_tags": ["N/A"],
                        "pred_tags": ["N/A"]}
                results.append(item)
                break
            
            pbar.set_description(f"Processing sentence {beg_idx}, fails: {fails_amt}")
            continue
        
        item = {"tokens": sample_input["tokens"], 
                "true_tags": sample_input["ner_tags"],
                "pred_tags": response_parsed}
        results.append(item)
        
        break
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for item in results:
            f.write(json.dumps(item) + "\n")   

0it [00:00, ?it/s]


In [11]:
llm_results = try_load_results_from_file(OUTPUT_FILE)
llm_results = [item for item in llm_results if item["true_tags"] != ["N/A"]]

print(f"Total sentences processed: {len(llm_results)}")

true_tags = [item["true_tags"] for item in llm_results]
pred_tags = [item["pred_tags"] for item in llm_results]

print("LLM NER Classification Report:")
print(classification_report(true_tags, pred_tags))

Total sentences processed: 500
LLM NER Classification Report:
              precision    recall  f1-score   support

         LOC       0.72      0.87      0.79       288
        MISC       0.70      0.58      0.64        65
         ORG       0.63      0.47      0.54       188
         PER       0.97      0.95      0.96       442

   micro avg       0.82      0.81      0.81       983
   macro avg       0.75      0.72      0.73       983
weighted avg       0.81      0.81      0.81       983



# Pytanie badawcze 1: Wpływ liczby i jakości przykładów (ICL)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

Q1_K_VALUES   = [0, 1, 2, 4, 8]
Q1_STRATEGIES = ["random", "entity_rich", "similar", "diverse"]
Q1_TEST_SUBSET = TEST_SUBSET
Q1_RESULTS_DIR = "q1_results"
os.makedirs(Q1_RESULTS_DIR, exist_ok=True)

PROMPT_TEMPLATE_WITH_EXAMPLES = """   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.

    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Examples:
    [EXAMPLE]

    Now process the following input.

    Input:
    [INPUT]
"""

PROMPT_TEMPLATE_ZERO_SHOT = """   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.

    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Now process the following input.

    Input:
    [INPUT]
"""

def format_examples(examples: list) -> str:
    parts = []
    for idx, ex in enumerate(examples):
        parts.append(
            f"        {idx+1})\n"
            f"        Input:\n"
            f"        {json.dumps(ex['tokens'])}\n"
            f"        Output:\n"
            f"        {json.dumps(ex['ner_tags'])}"
        )
    return "\n".join(parts)

def get_examples_random(k: int, **kwargs) -> str:
    if k == 0:
        return ""
    pool = list(range(len(train_dataset)))
    chosen = random.sample(pool, k)
    return format_examples([train_dataset[i] for i in chosen])

def get_examples_entity_rich(k: int, **kwargs) -> str:
    if k == 0:
        return ""
    scored = sorted(
        range(len(train_dataset)),
        key=lambda i: sum(1 for t in train_dataset[i]["ner_tags"] if t != "O"),
        reverse=True
    )
    examples = [train_dataset[i] for i in scored[:k]]
    return format_examples(examples)

def get_examples_similar(k: int, query_tokens: list, **kwargs) -> str:
    if k == 0:
        return ""
    query_set = set(t.lower() for t in query_tokens)
    scored = sorted(
        range(len(train_dataset)),
        key=lambda i: len(set(t.lower() for t in train_dataset[i]["tokens"]) & query_set),
        reverse=True
    )
    examples = [train_dataset[i] for i in scored[:k]]
    return format_examples(examples)

def get_examples_diverse(k: int, **kwargs) -> str:
    if k == 0:
        return ""
    entity_types = ["B-PER", "B-ORG", "B-LOC", "B-MISC"]
    used = set()
    examples = []
    type_cycle = (entity_types * ((k // 4) + 1))[:k]
    for etype in type_cycle:
        candidates = [i for i in range(len(train_dataset))
                      if etype in train_dataset[i]["ner_tags"] and i not in used]
        if candidates:
            chosen = random.choice(candidates)
            used.add(chosen)
            examples.append(train_dataset[chosen])
    if len(examples) < k:
        remaining = [train_dataset[i] for i in range(len(train_dataset)) if i not in used]
        examples += random.sample(remaining, k - len(examples))
    return format_examples(examples[:k])

STRATEGY_FN = {
    "random":       get_examples_random,
    "entity_rich":  get_examples_entity_rich,
    "similar":      get_examples_similar,
    "diverse":      get_examples_diverse,
}

def build_prompt_q1(example_str: str, input_tokens: list) -> str:
    if example_str:
        return (PROMPT_TEMPLATE_WITH_EXAMPLES
                .replace("[EXAMPLE]", example_str)
                .replace("[INPUT]", json.dumps(input_tokens)))
    return PROMPT_TEMPLATE_ZERO_SHOT.replace("[INPUT]", json.dumps(input_tokens))


In [ ]:
for k in Q1_K_VALUES:
    for strategy in Q1_STRATEGIES:
        config_name = f"k{k}_{strategy}"
        output_file = os.path.join(Q1_RESULTS_DIR, f"{config_name}.jsonl")

        results = try_load_results_from_file(output_file)
        if len(results) >= Q1_TEST_SUBSET:
            print(f"[{config_name}] already complete ({len(results)} sentences).")
            continue

        if strategy != "similar":
            random.seed(42)
            fixed_example_str = STRATEGY_FN[strategy](k=k)

        pbar = tqdm(range(len(results), Q1_TEST_SUBSET), desc=config_name)
        for idx in pbar:
            sample = test_dataset[idx]

            if strategy == "similar":
                example_str = get_examples_similar(k=k, query_tokens=sample["tokens"])
            else:
                example_str = fixed_example_str

            prompt = build_prompt_q1(example_str, sample["tokens"])

            fails = 0
            while True:
                response = LLM_API.send_message(
                    prompt,
                    is_reasoning=fails >= 2,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS * (fails + 1) if fails < 2 else None,
                    attempts_amt=15
                )
                pred, msg = parse_response(sample, response)
                if pred is None:
                    fails += 1
                    if fails >= MAX_FAILS_AMT:
                        results.append({"tokens": ["N/A"], "true_tags": ["N/A"], "pred_tags": ["N/A"]})
                        break
                    pbar.set_description(f"{config_name} [fail {fails}]")
                    continue
                results.append({"tokens": sample["tokens"],
                                "true_tags": sample["ner_tags"],
                                "pred_tags": pred})
                break

            with open(output_file, "w", encoding="utf-8") as f:
                for item in results:
                    f.write(json.dumps(item) + "\n")

print("Q1 experiment complete.")


k0_random [fail 4]:   1%|▏         | 1/74 [00:25<08:40,  7.13s/it]

Empty response, retrying... ({'id': 'gen-1778689475-IbeCmWsHOckaZbQUX26R', 'object': 'chat.completion', 'created': 1778689475, 'model': 'deepseek/deepseek-v4-flash-20260423', 'provider': 'AkashML', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': None, 'native_finish_reason': None, 'error': {'code': 502, 'message': 'Network connection lost.', 'metadata': {'error_type': 'provider_unavailable'}}, 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': 'We need to tag each token in the given list. The tokens are: "Extras", "(", "b-9", "lb-3", "w-12", "nb-2", ")", "26', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to tag each token in the given list. The tokens are: "Extras", "(", "b-9", "lb-3", "w-12", "nb-2", ")", "26', 'format': 'unknown', 'index': 0}]}}], 'usage': {'prompt_tokens': 146, 'completion_tokens': 29, 'total_tokens': 175, 'cost': 2.856e-05, 'is_byok': False, 'prompt_tokens_details': {'cached_t

k0_random [fail 1]:  34%|███▍      | 25/74 [11:29<20:45, 25.41s/it] 

Empty response, retrying... ({'id': 'gen-1778690138-u3JdNaqaznNXRJ4ClfLz', 'object': 'chat.completion', 'created': 1778690138, 'model': 'deepseek/deepseek-v4-flash-20260423', 'provider': 'SiliconFlow', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'length', 'native_finish_reason': 'length', 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': 'We need to tag each token in the input list with BIO tags. The input is a list of tokens. We need to identify named entities: persons, organizations, locations, miscellaneous. Let\'s go through tokens:\n\n"Leeds" - likely a location (city) or organization (football club). In context, "Leeds" is a city in England, but also could be Leeds United? The sentence: "Leeds \' England under-21 striker Lee Bowyer ..." Actually "Leeds" is part of "Leeds \' England"? Wait token list: ["Leeds", "\'", "England", "under-21", "striker", "Lee", "Bowyer", ...]. So "Leeds" is a location? But then 

k0_random [fail 3]:  39%|███▉      | 29/74 [15:05<33:25, 44.57s/it]

In [ ]:
q1_scores = {}
for k in Q1_K_VALUES:
    for strategy in Q1_STRATEGIES:
        config_name = f"k{k}_{strategy}"
        output_file  = os.path.join(Q1_RESULTS_DIR, f"{config_name}.jsonl")
        results = [r for r in try_load_results_from_file(output_file)
                   if r["true_tags"] != ["N/A"]
                   and isinstance(r["true_tags"], list)
                   and isinstance(r["pred_tags"], list)
                   and r["pred_tags"]
                   and isinstance(r["pred_tags"][0], str)]
        if not results:
            q1_scores[(k, strategy)] = None
            continue
        score = f1_score([r["true_tags"] for r in results],
                         [r["pred_tags"] for r in results],
                         average="micro")
        q1_scores[(k, strategy)] = round(score, 4)
        print(f"[k={k:2d}, {strategy:12s}] F1={score:.4f}  (n={len(results)})")

df_q1 = pd.DataFrame(
    [[q1_scores.get((k, s)) for s in Q1_STRATEGIES] for k in Q1_K_VALUES],
    index=[f"k={k}" for k in Q1_K_VALUES],
    columns=Q1_STRATEGIES
)
print("\nF1 Score Table:")
print(df_q1.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for strategy in Q1_STRATEGIES:
    vals = [q1_scores.get((k, strategy)) for k in Q1_K_VALUES]
    ax.plot(Q1_K_VALUES, vals, marker='o', label=strategy)
ax.set_xlabel("Liczba przykładów (k)")
ax.set_ylabel("Micro F1")
ax.set_title("F1 vs liczba przykładów — wg strategii")
ax.legend()
ax.grid(True)

ax2 = axes[1]
sns.heatmap(df_q1.astype(float), annot=True, fmt=".3f", cmap="YlGn",
            linewidths=0.5, vmin=0.5, vmax=1.0, ax=ax2)
ax2.set_title("Heatmapa F1: liczba × strategia")

plt.tight_layout()
plt.savefig("q1_results.png", dpi=150)
plt.show()

# Pytanie badawcze 2 (ICL): Wpływ pominięcia klasy z promptu na jakość predykcji

In [ ]:
Q2_ICL_OMIT_CLASSES = [None, "PER", "ORG", "LOC", "MISC"]
Q2_ICL_TEST_SUBSET  = TEST_SUBSET
Q2_ICL_EXAMPLES_K   = 2
Q2_ICL_RESULTS_DIR  = "q2_icl_results"
Q2_ICL_CLASS_NAMES  = ["LOC", "MISC", "ORG", "PER"]

os.makedirs(Q2_ICL_RESULTS_DIR, exist_ok=True)

_CLASS_TAG_LINES = {
    "PER":  "    - B-PER, I-PER\n",
    "ORG":  "    - B-ORG, I-ORG\n",
    "LOC":  "    - B-LOC, I-LOC\n",
    "MISC": "    - B-MISC, I-MISC\n",
}


def _fmt_examples_q2(examples: list) -> str:
    parts = []
    for idx, ex in enumerate(examples):
        parts.append(
            f"        {idx+1})\n"
            f"        Input:\n"
            f"        {json.dumps(ex['tokens'])}\n"
            f"        Output:\n"
            f"        {json.dumps(ex['ner_tags'])}"
        )
    return "\n".join(parts)


def get_examples_q2_icl(omit_class, k: int = Q2_ICL_EXAMPLES_K) -> str:
    drop = ({f"B-{omit_class}", f"I-{omit_class}"} if omit_class else set())
    pool = [i for i in range(len(train_dataset))
            if not any(t in drop for t in train_dataset[i]["ner_tags"])]
    chosen = random.sample(pool, min(k, len(pool)))
    return _fmt_examples_q2([train_dataset[i] for i in chosen])


def build_prompt_q2_icl(omit_class, example_str: str, input_tokens: list) -> str:
    tag_lines = "".join(v for k, v in _CLASS_TAG_LINES.items() if k != omit_class)
    example_section = f"    Examples:\n    {example_str}\n\n" if example_str else ""
    return (
        f"   You are a Named Entity Recognition system.\n\n"
        f"    Task: Tag each token using BIO format:\n"
        f"{tag_lines}"
        f"    - O for non-entities\n\n"
        f"    Rules:\n"
        f"    - Output ONLY valid JSON\n"
        f"    - Do not add explanations\n"
        f"    - If unsure, use \"O\".\n"
        f"    - Use english.\n\n"
        f"    Before returning:\n"
        f"    - Verify JSON is valid\n"
        f"    - Verify that sentences lengths match\n\n"
        f"    {example_section}"
        f"    Now process the following input.\n\n"
        f"    Input:\n"
        f"    {json.dumps(input_tokens)}\n"
    )


In [ ]:
random.seed(42)

for omit_class in Q2_ICL_OMIT_CLASSES:
    cfg         = f"no_{omit_class}" if omit_class else "baseline"
    output_file = os.path.join(Q2_ICL_RESULTS_DIR, f"{cfg}.jsonl")
    results     = try_load_results_from_file(output_file)

    if len(results) >= Q2_ICL_TEST_SUBSET:
        print(f"[{cfg}] already complete ({len(results)} sentences).")
        continue

    random.seed(42)
    fixed_example_str = get_examples_q2_icl(omit_class)

    pbar = tqdm(range(len(results), Q2_ICL_TEST_SUBSET), desc=cfg)
    for idx in pbar:
        sample = test_dataset[idx]
        prompt = build_prompt_q2_icl(omit_class, fixed_example_str, sample["tokens"])

        fails = 0
        while True:
            response = LLM_API.send_message(
                prompt,
                is_reasoning=fails >= 2,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS * (fails + 1) if fails < 2 else None,
                attempts_amt=15,
            )
            pred, _ = parse_response(sample, response)
            if pred is None:
                fails += 1
                if fails >= MAX_FAILS_AMT:
                    results.append({"tokens": ["N/A"], "true_tags": ["N/A"], "pred_tags": ["N/A"]})
                    break
                pbar.set_description(f"{cfg} [fail {fails}]")
                continue
            results.append({"tokens": sample["tokens"],
                            "true_tags": sample["ner_tags"],
                            "pred_tags": pred})
            break

        with open(output_file, "w", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item) + "\n")

print("Q2 ICL — wszystkie konfiguracje gotowe.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def per_class_f1_icl(results):
    results = [r for r in results if r["true_tags"] != ["N/A"]]
    if not results:
        return {cls: None for cls in Q2_ICL_CLASS_NAMES}
    rep = classification_report(
        [r["true_tags"] for r in results],
        [r["pred_tags"] for r in results],
        output_dict=True
    )
    return {cls: rep.get(cls, {}).get("f1-score", 0.0) for cls in Q2_ICL_CLASS_NAMES}

configs_q2_icl = [f"no_{c}" if c else "baseline" for c in Q2_ICL_OMIT_CLASSES]

q2_icl_scores = {}
for cfg in configs_q2_icl:
    output_file = os.path.join(Q2_ICL_RESULTS_DIR, f"{cfg}.jsonl")
    results     = try_load_results_from_file(output_file)
    q2_icl_scores[cfg] = per_class_f1_icl(results)
    n = sum(1 for r in results if r["true_tags"] != ["N/A"])
    print(f"[{cfg}] n={n}  " +
          "  ".join(f"{cls}={q2_icl_scores[cfg][cls]:.3f}" for cls in Q2_ICL_CLASS_NAMES))

df_q2_icl = pd.DataFrame(q2_icl_scores).T[Q2_ICL_CLASS_NAMES]

print("\nPer-class F1 (ICL Q2):")
print(df_q2_icl.to_string())

df_delta_icl = df_q2_icl.drop("baseline") - df_q2_icl.loc["baseline"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(
    df_q2_icl.astype(float), annot=True, fmt=".3f",
    cmap="RdYlGn", linewidths=0.5, vmin=0.4, vmax=1.0,
    ax=axes[0]
)
axes[0].set_title("ICL Q2: Per-class F1\n(wiersze = pomięta klasa z promptu)")
axes[0].set_xlabel("Klasa oceniana")
axes[0].set_ylabel("Konfiguracja promptu")

sns.heatmap(
    df_delta_icl.astype(float), annot=True, fmt="+.3f",
    cmap="RdBu", center=0, linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title("ICL Q2: Zmiana F1 vs baseline\n(+ poprawa, − pogorszenie)")
axes[1].set_xlabel("Klasa oceniana")
axes[1].set_ylabel("Pominięta klasa")

plt.tight_layout()
plt.savefig("q2_icl_results.png", dpi=150)
plt.show()


# Pytanie badawcze 3 (ICL): Ograniczenie promptu do pojedynczej klasy encji

In [ ]:
Q3_ICL_KEEP_CLASSES = ["PER", "ORG", "LOC", "MISC"]
Q3_ICL_TEST_SUBSET  = TEST_SUBSET
Q3_ICL_EXAMPLES_K   = 2
Q3_ICL_RESULTS_DIR  = "q3_icl_results"
Q3_ICL_CLASS_NAMES  = ["LOC", "MISC", "ORG", "PER"]

os.makedirs(Q3_ICL_RESULTS_DIR, exist_ok=True)


def get_examples_q3_icl(keep_class: str, k: int = Q3_ICL_EXAMPLES_K) -> str:
    target = {f"B-{keep_class}", f"I-{keep_class}"}
    pool = [i for i in range(len(train_dataset))
            if any(t in target for t in train_dataset[i]["ner_tags"])]
    chosen = random.sample(pool, min(k, len(pool)))
    return _fmt_examples_q2([train_dataset[i] for i in chosen])


def build_prompt_q3_icl(keep_class: str, example_str: str, input_tokens: list) -> str:
    tag_line = _CLASS_TAG_LINES[keep_class] 
    example_section = f"    Examples:\n    {example_str}\n\n" if example_str else ""
    return (
        f"   You are a Named Entity Recognition system.\n\n"
        f"    Task: Tag each token using BIO format:\n"
        f"{tag_line}"
        f"    - O for all other tokens\n\n"
        f"    Rules:\n"
        f"    - Output ONLY valid JSON\n"
        f"    - Do not add explanations\n"
        f"    - If unsure, use \"O\".\n"
        f"    - Use english.\n\n"
        f"    Before returning:\n"
        f"    - Verify JSON is valid\n"
        f"    - Verify that sentences lengths match\n\n"
        f"    {example_section}"
        f"    Now process the following input.\n\n"
        f"    Input:\n"
        f"    {json.dumps(input_tokens)}\n"
    )


In [ ]:
q3_icl_results = {}
_q2_baseline_file = os.path.join(Q2_ICL_RESULTS_DIR, "baseline.jsonl")
if os.path.exists(_q2_baseline_file):
    q3_icl_results["baseline"] = try_load_results_from_file(_q2_baseline_file)
    print(f"Baseline: załadowano z Q2 ({len(q3_icl_results['baseline'])} zdań).")

random.seed(42)

for keep_class in Q3_ICL_KEEP_CLASSES:
    cfg         = f"only_{keep_class}"
    output_file = os.path.join(Q3_ICL_RESULTS_DIR, f"{cfg}.jsonl")
    results     = try_load_results_from_file(output_file)

    if len(results) >= Q3_ICL_TEST_SUBSET:
        print(f"[{cfg}] already complete ({len(results)} sentences).")
        q3_icl_results[cfg] = results
        continue

    random.seed(42)
    fixed_example_str = get_examples_q3_icl(keep_class)

    pbar = tqdm(range(len(results), Q3_ICL_TEST_SUBSET), desc=cfg)
    for idx in pbar:
        sample = test_dataset[idx]
        prompt = build_prompt_q3_icl(keep_class, fixed_example_str, sample["tokens"])

        fails = 0
        while True:
            response = LLM_API.send_message(
                prompt,
                is_reasoning=fails >= 2,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS * (fails + 1) if fails < 2 else None,
                attempts_amt=15,
            )
            pred, _ = parse_response(sample, response)
            if pred is None:
                fails += 1
                if fails >= MAX_FAILS_AMT:
                    results.append({"tokens": ["N/A"], "true_tags": ["N/A"], "pred_tags": ["N/A"]})
                    break
                pbar.set_description(f"{cfg} [fail {fails}]")
                continue
            results.append({"tokens": sample["tokens"],
                            "true_tags": sample["ner_tags"],
                            "pred_tags": pred})
            break

        with open(output_file, "w", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item) + "\n")

    q3_icl_results[cfg] = results

print("Q3 ICL — wszystkie konfiguracje gotowe.")


In [ ]:
def per_class_f1_q3(results):
    results = [r for r in results if r["true_tags"] != ["N/A"]]
    if not results:
        return {cls: 0.0 for cls in Q3_ICL_CLASS_NAMES}
    rep = classification_report(
        [r["true_tags"] for r in results],
        [r["pred_tags"] for r in results],
        output_dict=True,
    )
    return {cls: rep.get(cls, {}).get("f1-score", 0.0) for cls in Q3_ICL_CLASS_NAMES}

configs_q3_icl = ["baseline"] + [f"only_{c}" for c in Q3_ICL_KEEP_CLASSES]

q3_icl_scores = {}
for cfg in configs_q3_icl:
    if cfg in q3_icl_results:
        q3_icl_scores[cfg] = per_class_f1_q3(q3_icl_results[cfg])
        n = sum(1 for r in q3_icl_results[cfg] if r["true_tags"] != ["N/A"])
        print(f"[{cfg}] n={n}  " +
              "  ".join(f"{cls}={q3_icl_scores[cfg][cls]:.3f}" for cls in Q3_ICL_CLASS_NAMES))

df_q3_icl = pd.DataFrame(q3_icl_scores).T[Q3_ICL_CLASS_NAMES]
print("\nPer-class F1 (ICL Q3):")
print(df_q3_icl.to_string())

df_delta_q3_icl = df_q3_icl.drop("baseline") - df_q3_icl.loc["baseline"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(
    df_q3_icl.astype(float), annot=True, fmt=".3f",
    cmap="RdYlGn", linewidths=0.5, vmin=0.0, vmax=1.0,
    ax=axes[0]
)
axes[0].set_title("ICL Q3: Per-class F1\n(wiersze = jedyna klasa w prompcie)")
for i, cls in enumerate(Q3_ICL_CLASS_NAMES):
    cfg = f"only_{cls}"
    if cfg in configs_q3_icl:
        row_idx = configs_q3_icl.index(cfg)
        col_idx = Q3_ICL_CLASS_NAMES.index(cls)
        axes[0].add_patch(
            plt.Rectangle((col_idx, row_idx), 1, 1,
                           fill=False, edgecolor="blue", lw=2)
        )
axes[0].set_xlabel("Klasa oceniana")
axes[0].set_ylabel("Konfiguracja promptu")

sns.heatmap(
    df_delta_q3_icl.astype(float), annot=True, fmt="+.3f",
    cmap="RdBu", center=0, linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title("ICL Q3: Zmiana F1 vs baseline\n(+ poprawa, − pogorszenie)")
axes[1].set_xlabel("Klasa oceniana")
axes[1].set_ylabel("Jedyna klasa w prompcie")

plt.tight_layout()
plt.savefig("q3_icl_results.png", dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(7, 4))
baseline_f1_icl = [df_q3_icl.loc["baseline", cls] for cls in Q3_ICL_CLASS_NAMES]
only_f1_icl     = [df_q3_icl.loc[f"only_{cls}", cls] for cls in Q3_ICL_CLASS_NAMES]

x = range(len(Q3_ICL_CLASS_NAMES))
w = 0.35
ax2.bar([i - w/2 for i in x], baseline_f1_icl, w, label="baseline", color="steelblue")
ax2.bar([i + w/2 for i in x], only_f1_icl,     w, label="only_X",   color="coral")
ax2.set_xticks(list(x))
ax2.set_xticklabels(Q3_ICL_CLASS_NAMES)
ax2.set_ylabel("F1")
ax2.set_ylim(0, 1)
ax2.set_title("ICL Q3: Specjalizacja — F1 klasy X gdy prompt zawiera tylko X")
ax2.legend()
ax2.grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("q3_icl_specialization.png", dpi=150)
plt.show()


# Pytanie badawcze 4: Porównanie wszystkich konfiguracji — najlepsza jakość na 4 klasach

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

_CLASS_COLS  = ["LOC_f1", "MISC_f1", "ORG_f1", "PER_f1"]
_CLASS_NAMES = ["LOC", "MISC", "ORG", "PER"]


def _score_results(results_list):
    valid = [r for r in results_list if r["true_tags"] != ["N/A"]]
    if not valid:
        return None
    rep = classification_report(
        [r["true_tags"] for r in valid],
        [r["pred_tags"] for r in valid],
        output_dict=True,
    )
    return {
        "micro_f1":    round(rep.get("micro avg", {}).get("f1-score", 0.0), 4),
        "macro_f1":    round(rep.get("macro avg", {}).get("f1-score", 0.0), 4),
        "LOC_f1":      round(rep.get("LOC",  {}).get("f1-score", 0.0), 4),
        "MISC_f1":     round(rep.get("MISC", {}).get("f1-score", 0.0), 4),
        "ORG_f1":      round(rep.get("ORG",  {}).get("f1-score", 0.0), 4),
        "PER_f1":      round(rep.get("PER",  {}).get("f1-score", 0.0), 4),
        "n_sentences": len(valid),
    }


all_q4_rows = []

_bert_file = "bert_baseline_predictions.json"
if os.path.exists(_bert_file):
    with open(_bert_file) as f:
        _bd = json.load(f)
    _bert_valid = [{"true_tags": tl, "pred_tags": tp}
                   for tl, tp in zip(_bd["true_tags"], _bd["pred_tags"])]
    _s = _score_results(_bert_valid)
    if _s:
        all_q4_rows.append({"config": "BERT_baseline", "model": "BERT", **_s})
else:
    print("WARN: bert_baseline_predictions.json nie znaleziony — uruchom najpierw BERT notebook Q4.")

for fname, label in [(OUTPUT_FILE, f"LLM_orig_k2_random ({OUTPUT_FILE.split('_')[0]})")]:
    _r = try_load_results_from_file(fname)
    _s = _score_results(_r)
    if _s:
        all_q4_rows.append({"config": label, "model": "LLM", **_s})

if os.path.exists(Q1_RESULTS_DIR):
    for k in Q1_K_VALUES:
        for strategy in Q1_STRATEGIES:
            cfg  = f"k{k}_{strategy}"
            path = os.path.join(Q1_RESULTS_DIR, f"{cfg}.jsonl")
            _r   = try_load_results_from_file(path)
            _s   = _score_results(_r)
            if _s:
                all_q4_rows.append({"config": f"LLM_Q1_{cfg}", "model": "LLM_Q1", **_s})

_q2_base = os.path.join(Q2_ICL_RESULTS_DIR, "baseline.jsonl")
_r = try_load_results_from_file(_q2_base)
_s = _score_results(_r)
if _s:
    all_q4_rows.append({"config": "LLM_Q2_baseline", "model": "LLM_Q2", **_s})

_q3_base = os.path.join(Q3_ICL_RESULTS_DIR, "baseline.jsonl")
_r = try_load_results_from_file(_q3_base)
_s = _score_results(_r)
if _s:
    all_q4_rows.append({"config": "LLM_Q3_baseline", "model": "LLM_Q3", **_s})

df_q4 = pd.DataFrame(all_q4_rows).sort_values("micro_f1", ascending=False).reset_index(drop=True)
df_q4.index += 1

print(f"Łącznie konfiguracji: {len(df_q4)}")
print("\nRanking wg micro F1:")
print(df_q4[["config", "model", "micro_f1", "macro_f1"] + _CLASS_COLS + ["n_sentences"]].to_string())


In [ ]:
TOP_N = min(15, len(df_q4))
df_top = df_q4.head(TOP_N)

model_colors = {
    "BERT":    "steelblue",
    "LLM":     "coral",
    "LLM_Q1":  "mediumseagreen",
    "LLM_Q2":  "mediumpurple",
    "LLM_Q3":  "goldenrod",
}

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, TOP_N * 0.45 + 1)))

ax = axes[0]
colors = [model_colors.get(m, "gray") for m in df_top["model"]]
bars = ax.barh(range(TOP_N), df_top["micro_f1"].values, color=colors)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(df_top["config"].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Micro F1")
ax.set_title(f"Top-{TOP_N} konfiguracji wg micro F1")
ax.set_xlim(0, 1)
ax.axvline(df_q4["micro_f1"].max(), color="red", linestyle="--", linewidth=0.8, alpha=0.6)

for i, (val, n) in enumerate(zip(df_top["micro_f1"], df_top["n_sentences"])):
    ax.text(val + 0.005, i, f"{val:.3f} (n={n})", va="center", fontsize=7)

legend_patches = [mpatches.Patch(color=c, label=m) for m, c in model_colors.items()]
ax.legend(handles=legend_patches, loc="lower right", fontsize=8)

ax2 = axes[1]
heatmap_data = df_top[_CLASS_COLS].values.astype(float)
hm = ax2.imshow(heatmap_data, aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=1.0)
ax2.set_xticks(range(len(_CLASS_NAMES)))
ax2.set_xticklabels(_CLASS_NAMES)
ax2.set_yticks(range(TOP_N))
ax2.set_yticklabels(df_top["config"].values, fontsize=8)
ax2.set_title(f"Per-class F1 — top {TOP_N} konfiguracji")
plt.colorbar(hm, ax=ax2, fraction=0.03)

for i in range(TOP_N):
    for j in range(len(_CLASS_NAMES)):
        val = heatmap_data[i, j]
        ax2.text(j, i, f"{val:.2f}", ha="center", va="center",
                 fontsize=7, color="black" if val > 0.5 else "white")

plt.tight_layout()
plt.savefig("q4_final_ranking.png", dpi=150)
plt.show()

best = df_q4.iloc[0]
print(f"\n{'='*60}")
print(f"Najlepsza konfiguracja: {best['config']}")
print(f"  Model:      {best['model']}")
print(f"  Micro F1:   {best['micro_f1']:.4f}")
print(f"  Macro F1:   {best['macro_f1']:.4f}")
print(f"  LOC F1:     {best['LOC_f1']:.4f}")
print(f"  MISC F1:    {best['MISC_f1']:.4f}")
print(f"  ORG F1:     {best['ORG_f1']:.4f}")
print(f"  PER F1:     {best['PER_f1']:.4f}")
print(f"  Oceniono:   {best['n_sentences']} zdań")
print(f"{'='*60}")

bert_row  = df_q4[df_q4["model"] == "BERT"]
llm_rows  = df_q4[df_q4["model"].str.startswith("LLM")]
if not bert_row.empty and not llm_rows.empty:
    best_llm = llm_rows.iloc[0]
    print(f"\nBERT baseline micro F1:    {bert_row.iloc[0]['micro_f1']:.4f}")
    print(f"Najlepszy LLM micro F1:    {best_llm['micro_f1']:.4f}  ({best_llm['config']})")
    diff = bert_row.iloc[0]["micro_f1"] - best_llm["micro_f1"]
    winner = "BERT" if diff > 0 else "LLM"
    print(f"Przewaga {winner}: {abs(diff):.4f}")
